# Cross-model self-recognition — Llama vs Qwen

**Question.** When a model judges *"which of these two texts did I write?"*, is it
recognizing its **own model** (Llama vs Qwen) or its **own persona** (the specific
system prompt it was under)? We separate the two.

Each trial pairs the evaluator's **self** text — generated by *(evaluator_model,
evaluator_persona)* — against one **foil** on the same task. The foil differs from
the evaluator along one or both axes (`foil_type`):

- **`diff_model_same_persona`** — other model, *same* persona → **model recognition**
  (e.g. Llama-neutral vs Qwen-neutral: can Llama pick its own?).
- **`same_model_diff_persona`** — same model, *different* persona → within-model
  persona recognition.
- **`diff_model_diff_persona`** — both differ → easiest.

Two conditions: **active** (evaluator persona induced as a system prompt) and
**neutral** (no system prompt — the baseline that isolates the active-state gain).
Text order and answer letter are counterbalanced, so **chance = 0.5** and raw
accuracy is position/letter-bias free. Rows are `PersonaCrossModelEvalRecord`
(see `core/schemas.py`), produced by `evaluate_cross_model.py`
(config: `self_recognition_cross_model`).

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / "core").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import logging; logging.basicConfig(level=logging.INFO)
import pandas as pd, matplotlib.pyplot as plt
import experiments.self_recognition.analyze_behavior_helpers as B      # stats + plotters
import experiments.self_recognition.analyze_cross_model_helpers as X   # cross-model load/agg
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

EVAL_DIR = "cross_model_v1"                       # the config's eval_dir
MODELS = ["meta-llama/Llama-3.1-8B-Instruct", "Qwen/Qwen2.5-32B-Instruct"]
df = X.load_cross_model(EVAL_DIR, MODELS)
print(f"{len(df):,} trials | evaluators: {sorted(df.evaluator_slug.unique())}")
print("conditions:", df.condition.value_counts().to_dict())
print("foil types:", df.foil_type.value_counts().to_dict())

## 0 · Balance check

Counterbalancing should make the correct answer ≈ 50% A and the self text ≈ 50%
first within every (evaluator, foil_type, condition) cell. If not, something is
wrong with enumeration.

In [ ]:
bal = (df.assign(correctA=df.correct_answer.eq("A"), self1=df.text1_is_self)
         .groupby(["evaluator_slug","foil_type","condition"])
         .agg(frac_correct_A=("correctA","mean"), frac_self_first=("self1","mean"), n=("trial_id","size")))
display(bal.round(3))

## 1 · Model recognition — can a model pick its own output from the other model's?

Restricted to **`diff_model_same_persona`** (foil = other model, *same* persona),
so the ONLY thing that differs between the two texts is which model wrote them.
Above 0.5 = the model can recognize its own model-level fingerprint. Compare
**active vs neutral**: if neutral is already high, the model-fingerprint is in the
text itself (doesn't need the persona active).

In [ ]:
mr = X.model_recognition(df, by=("evaluator_slug","condition"))
mr["label"] = mr.evaluator_slug + "\n" + mr.condition
display(mr[["evaluator_slug","condition","n","accuracy","ci_lo","ci_hi","p_vs_chance","mean_prob_correct"]])
B.plot_accuracy(mr, title="Model recognition (diff_model_same_persona) — own vs other model")
plt.show()
B.plot_accuracy(mr, value="mean_prob_correct", lo_col="prob_ci_lo", hi_col="prob_ci_hi",
                ylabel="mean P(correct)", color="#55A868",
                title="Model recognition — graded logprob mass")
plt.show()

### 1b · Model recognition on the NEUTRAL persona specifically

Your first question: *can Llama and Qwen recognize their own **neutral** texts from
the other model?* This is model recognition with `evaluator_persona = default_neutral`
(no persona style to lean on — pure model fingerprint).

In [ ]:
NEUTRAL_PERSONA = "default_neutral"   # adjust if your neutral persona is named differently
sub = df[(df.foil_type=="diff_model_same_persona") & (df.evaluator_persona==NEUTRAL_PERSONA)]
if sub.empty:
    print(f"no rows for evaluator_persona={NEUTRAL_PERSONA!r}; personas present:",
          sorted(df.evaluator_persona.unique()))
else:
    nt = X.agg(sub, by=["evaluator_slug","condition"])
    nt["label"] = nt.evaluator_slug + "\n" + nt.condition
    display(nt[["evaluator_slug","condition","n","accuracy","ci_lo","ci_hi","p_vs_chance","mean_prob_correct"]])
    B.plot_accuracy(nt, title=f"Neutral-persona model recognition ({NEUTRAL_PERSONA})")
    plt.show()

## 2 · Foil-type ladder — model cue vs persona cue

For the **active** persona, how hard is each foil to reject? Read within an
evaluator model:

- **other model, same persona** — only the model differs. High here = strong
  *model* self-recognition.
- **same model, other persona** — only the persona differs. High here = strong
  *persona* self-recognition.
- **other model, other persona** — both differ (should be easiest).

The contrast between the first two tells you whether the model's "self" is
anchored more to its **model identity** or its **persona identity**.

In [ ]:
ladder = X.foil_type_ladder(df, condition="active", by_model=True)
display(ladder[["evaluator_slug","foil_type","n","accuracy","ci_lo","ci_hi","p_vs_chance","mean_prob_correct"]])
B.plot_accuracy(ladder, title="Foil-type ladder (active persona) — model cue vs persona cue")
plt.show()

# By-category view: split each foil_type by the evaluator persona's category.
lc = X.agg(df[df.condition=="active"], by=["foil_label","evaluator_coarse"])
lc = lc.rename(columns={"foil_label":"label"})
if not lc.empty:
    B.plot_contrast_by_category(lc, title="Foil-type ladder by persona category (active)")
    plt.show()

## 3 · Active vs neutral surplus (the active-state gain)

Cross-model analogue of case7 − case10: how much does *inducing the persona* add
over the no-persona baseline, per foil_type? A positive surplus means being the
persona helps recognition beyond whatever is readable from the text alone.

In [ ]:
for ft in ["diff_model_same_persona","same_model_diff_persona","diff_model_diff_persona"]:
    s = X.active_vs_neutral(df, foil_type=ft, by=("evaluator_slug",))
    print(f"\n=== {ft} ===")
    display(s.round(3))

## 4 · Per-persona grid (evaluator model × persona)

Accuracy for the **model-recognition** foil (other model, same persona), active
condition, per persona and evaluator. Personas whose *style* is model-invariant
should still be recognizable here only via the model fingerprint — a clean look at
where model self-recognition is strong or absent.

In [ ]:
grid = X.agg(df[(df.foil_type=="diff_model_same_persona") & (df.condition=="active")],
             by=["evaluator_slug","evaluator_persona"])
if not grid.empty:
    piv = grid.pivot_table(index="evaluator_persona", columns="evaluator_slug",
                           values="accuracy", observed=True).round(3)
    display(piv)